# Comparación baseline full vs máscara (RoCoLe)

Comparar los baselines con foto completa y con máscara COCO sobre las mismas 226 hojas de prueba, leyendo solo lo ya guardado, para dejar claro qué mide la columna test y si la máscara realmente mejora.


## Semilla fija

Fijamos la semilla del proyecto para que cualquier azar futuro se pueda repetir.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Image, display
from pathlib import Path


In [ ]:
seed = 42

seed


## Rutas de tablas y resultados

Localizamos la tabla full, la tabla con máscara, el índice de hojas y la carpeta de resultados, todas relativas al repo.


In [ ]:
full_features_path = Path("../../data/featured/rocole/rgb_glcm_features.csv")
masked_features_path = Path("../../data/featured/rocole/rgb_glcm_features_masked.csv")
index_path = Path("../../data/featured/rocole/index.csv")
results_dir = Path("../../experiments/results")

print("full_features_path:", full_features_path.resolve(), "existe:", full_features_path.exists())
print("masked_features_path:", masked_features_path.resolve(), "existe:", masked_features_path.exists())
print("index_path:", index_path.resolve(), "existe:", index_path.exists())
print("results_dir:", results_dir.resolve(), "existe:", results_dir.exists())
full_features_path, masked_features_path, index_path, results_dir


## Cargar características full

Leemos la tabla de medidas sobre hoja completa. Conservamos File, split, link_group y Binary.Label; no inventamos un reparto.


In [ ]:
full_features = pd.read_csv(full_features_path)

print("filas:", len(full_features))
print("columnas:", list(full_features.columns))
full_features.head()


## Cargar características con máscara

Leemos la tabla de las mismas medidas, pero solo dentro de la máscara de hoja. Misma estructura que full.


In [ ]:
# masked_features_path

masked_features = pd.read_csv(masked_features_path)

print("filas:", len(masked_features))
print("columnas:", list(masked_features.columns))
masked_features.head()


## Cargar índice de hojas

Leemos el índice featured con File, split, link_group y Binary.Label. Es la referencia del reparto train/val/test.


In [ ]:
# index_path

leaf_index = pd.read_csv(index_path)

print("filas:", len(leaf_index))
print("columnas:", list(leaf_index.columns))
leaf_index.head()


## ¿Full y máscara van en el mismo orden?

Comprobamos que File, split, link_group y Binary.Label coinciden fila a fila entre full, máscara e índice, y que hay 1481 filas. Si falla, la comparación de baselines no es válida.


In [ ]:
# full_features, masked_features, leaf_index
meta_cols = ["File", "split", "link_group", "Binary.Label"]

full_meta = full_features[meta_cols].reset_index(drop=True)
masked_meta = masked_features[meta_cols].reset_index(drop=True)
index_meta = leaf_index[meta_cols].reset_index(drop=True)

alignment_ok = (
    len(full_features) == 1481
    and len(masked_features) == 1481
    and len(leaf_index) == 1481
    and full_meta.equals(masked_meta)
    and full_meta.equals(index_meta)
)

print("n full:", len(full_features), "n masked:", len(masked_features), "n index:", len(leaf_index))
print("alignment_ok:", alignment_ok)
alignment_ok


## ¿Algún link_group mezcla splits?

Agrupamos por link_group y contamos cuántos valores distintos de split aparecen. Ningún grupo debería tener más de uno: la misma planta no puede estar en aprender y en prueba.


In [ ]:
# leaf_index

splits_per_link_group = leaf_index.groupby("link_group")["split"].nunique()
link_group_split_ok = bool((splits_per_link_group <= 1).all())

print("link_groups con más de un split:", int((splits_per_link_group > 1).sum()))
print("link_group_split_ok:", link_group_split_ok)
link_group_split_ok


## Conteos por split en full y máscara

Contamos cuántas hojas caen en train, val y test en cada tabla. Deben coincidir: el reparto es el mismo.


In [ ]:
# full_features, masked_features

split_counts = pd.DataFrame(
    {
        "full": full_features["split"].value_counts(),
        "masked": masked_features["split"].value_counts(),
    }
).sort_index()

print(split_counts)
split_counts


## Etiquetas en el holdout de test

En las hojas con split igual a test, contamos healthy frente a unhealthy en ambas tablas. Esperamos 115 sanas y 111 afectadas (226 en total).


In [ ]:
# full_features, masked_features

full_test_labels = full_features.loc[full_features["split"] == "test", "Binary.Label"]
masked_test_labels = masked_features.loc[masked_features["split"] == "test", "Binary.Label"]
test_label_counts = pd.DataFrame(
    {
        "full": full_test_labels.value_counts(),
        "masked": masked_test_labels.value_counts(),
    }
).sort_index()

print("n test full:", len(full_test_labels), "n test masked:", len(masked_test_labels))
print("esperado: healthy 115, unhealthy 111")
print(test_label_counts)
test_label_counts


## Qué significa train, val y test aquí

El índice featured ya apartó las hojas en tres grupos, por planta (link_group), para que la misma planta no aparezca en dos lados.

- **train** (1037 hojas): ahí se ajustaron los modelos en los cuadernos 04. Solo se usa para aprender pesos o árboles.
- **val** (218 hojas): ahí se eligieron hiperparámetros (C de la logística, n_estimators y max_depth del bosque). No es el número que reportamos como acierto final.
- **test** (226 hojas): holdout. Se mira **una sola vez** al final de cada 04, con el f1_macro de test. No se usa para elegir columnas, umbrales ni hiperparámetros.

Este cuaderno 05 no vuelve a entrenar ni a tocar test para decidir nada: solo lee las métricas y matrices que ya salieron de esos holdouts.


## Métricas de test del baseline full

Cargamos el CSV de métricas de test del baseline sin máscara (ya guardado por el 04 full). No recalculamos f1.


In [ ]:
# results_dir

full_metrics_path = results_dir / "rocole_baseline_metrics.csv"
full_metrics = pd.read_csv(full_metrics_path)

print("ruta:", full_metrics_path.resolve(), "existe:", full_metrics_path.exists())
full_metrics


## Métricas de test del baseline con máscara

Cargamos el CSV de métricas de test del baseline con máscara (ya guardado por el 04 masked). Tampoco recalculamos.


In [ ]:
# results_dir

masked_metrics_path = results_dir / "rocole_baseline_masked_metrics.csv"
masked_metrics = pd.read_csv(masked_metrics_path)

print("ruta:", masked_metrics_path.resolve(), "existe:", masked_metrics_path.exists())
masked_metrics


## Comparación en val del baseline full

Leemos la tabla de f1_macro en val (mayoría, logística, bosque) del corrido full. Sirve para recordar qué ganó en validación, no como acierto final.


In [ ]:
# results_dir

full_val_path = results_dir / "rocole_baseline_val_comparison.csv"
full_val_comparison = pd.read_csv(full_val_path)

print("ruta:", full_val_path.resolve(), "existe:", full_val_path.exists())
full_val_comparison


## Comparación en val del baseline con máscara

Leemos la tabla de f1_macro en val del corrido con máscara. Misma lectura: val eligió; test ya se midió aparte.


In [ ]:
# results_dir

masked_val_path = results_dir / "rocole_baseline_masked_val_comparison.csv"
masked_val_comparison = pd.read_csv(masked_val_path)

print("ruta:", masked_val_path.resolve(), "existe:", masked_val_path.exists())
masked_val_comparison


## Tabla lado a lado: test full vs máscara

Unimos por modelo el f1_macro de test de full y de máscara (y el f1_macro de val como contexto). Esperado aproximado en test: full log 0.56 / RF 0.55; masked log 0.61 / RF 0.68. Sin reentrenar.


In [ ]:
# full_metrics, masked_metrics, full_val_comparison, masked_val_comparison

full_side = full_metrics.add_suffix("_full").rename(columns={"modelo_full": "modelo"})
masked_side = masked_metrics.add_suffix("_masked").rename(columns={"modelo_masked": "modelo"})
metrics_comparison = full_side.merge(masked_side, on="modelo", how="outer")

full_val = full_val_comparison.rename(columns={"f1_macro_val": "f1_macro_val_full"})
masked_val = masked_val_comparison.rename(columns={"f1_macro_val": "f1_macro_val_masked"})
metrics_comparison = metrics_comparison.merge(full_val, on="modelo", how="left")
metrics_comparison = metrics_comparison.merge(masked_val, on="modelo", how="left")

print(metrics_comparison.to_string(index=False))
metrics_comparison


## Matrices de confusión de test (números del 04)

Documentamos las matrices ya obtenidas en los cuadernos 04 (full y masked), sin recalcularlas. Filas = verdad (healthy, unhealthy); columnas = predicción (healthy, unhealthy). Números verificados: - full logística: [[61, 54], [45, 66]] - full bosque: [[59, 56], [46, 65]] - masked logística: [[69, 46], [43, 68]] - masked bosque: [[78, 37], [36, 75]]


In [ ]:
# Números tomados de los 04 ya corridos; no se recalculan aquí.
confusion_tables = {
    "full_logistic": np.array([[61, 54], [45, 66]]),
    "full_rf": np.array([[59, 56], [46, 65]]),
    "masked_logistic": np.array([[69, 46], [43, 68]]),
    "masked_rf": np.array([[78, 37], [36, 75]]),
}

for name, matrix in confusion_tables.items():
    print(name)
    print(matrix)
    print("filas=verdad healthy/unhealthy; cols=pred healthy/unhealthy")
    print()
confusion_tables


## Figuras de confusión ya guardadas

Listamos las cuatro PNG de matrices de confusión de test ya guardadas en results. Si el archivo existe, lo mostramos con Image; si no, solo avisamos la ruta. No regeneramos las figuras.


In [ ]:
confusion_figure_paths = {
    "full_logistic": results_dir / "rocole_baseline_logistic_confusion.png",
    "full_rf": results_dir / "rocole_baseline_rf_confusion.png",
    "masked_logistic": results_dir / "rocole_baseline_masked_logistic_confusion.png",
    "masked_rf": results_dir / "rocole_baseline_masked_rf_confusion.png",
}

for name, path in confusion_figure_paths.items():
    print(name, "->", path.resolve(), "existe:", path.exists())
    if path.exists():
        display(Image(filename=str(path)))
confusion_figure_paths


## Conclusiones

Misma base de hojas (226 en test): full y masked miden distinto sobre las mismas fotos. La máscara mejora el f1_macro de test, sobre todo en el random forest. El número de test es holdout, no acierto de entrenamiento. Este cuaderno no reentrenó ni rehizo splits: solo leyó lo ya guardado.
